# ATM 407: anatomy of an atmospheric column model

This lab treats the SCM as the physics column that would sit inside a dynamical core. You will examine the vertical coordinate, diagnose static stability, follow the sequence of parameterized tendencies, and test sensitivity to resolution and mass-flux convection parameters.

The SCM has no horizontal pressure-gradient force, advection, Coriolis acceleration, or resolved vertical motion. Keep that limitation in mind when connecting these results to atmospheric dynamics.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/02_experiments_atm407.ipynb)

## Colab setup

Run this cell first. It installs the current model and verifies that Python can import it.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

incolab = 'google.colab' in sys.modules
if incolab:
    root = Path('/content/GCM')
    if not root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/evanwellmeyer/GCM.git', str(root),
        ], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'pyproject.toml').exists():
        root = root.parent

if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from inside the GCM repository.')

from importlib import metadata, util

try:
    metadata.version('gcm-scm')
    installed = util.find_spec('matplotlib') is not None
except metadata.PackageNotFoundError:
    installed = False

if not installed:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{root}[plot]',
    ], check=True)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import scm
print('SCM ready from', Path(scm.__file__).resolve())

In [ ]:
from copy import deepcopy
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from scm.column_model import initial_state, physics_step, run, update_derived
from scm.configuration import extract_param_overrides, load_run_config
from scm.ensemble import default_params
from scm.thermo import Rd, cp, g, make_grid, relative_humidity

torch.manual_seed(0)
device = torch.device('cpu')
print('device:', device)

## Model controls

The notebook loads the accepted `mf_resolution_tuned_v1` 20-level reference state generated with multiband radiation and mass-flux convection. Its late-window TOA imbalance is 0.04 W m$^{-2}$, surface imbalance is -0.59 W m$^{-2}$, and 50-day temperature drift is 0.012 K. Energy and water budgets close tightly. The mass-flux cap remains active, however, so the reference is an equilibrated teaching state rather than evidence that every convection parameter has been calibrated. When another resolution is requested, temperature and water fields are interpolated in sigma coordinates and column water is conserved; the result is a balanced initial guess, not an equilibrium on the new grid.

In [ ]:
experiment = {
    'nlevels': 20,
    'dt': 900.0,
    'days': 3,
    'diagnostic_hours': 3,
    'radiation_steps': 8,
    'surface_temperature': 290.0,
    'surface_pressure': 100000.0,
    'solar_constant': 1360.0,
    'zenith_factor': 0.25,
    'ocean_depth': 50.0,
    'surface_albedo': 0.32,
    'wind_speed': 5.0,
}

referencemetadata = json.loads(
    (root / 'notebooks/data/atm407_equilibrium_20level.json').read_text()
)
print('reference configuration:', referencemetadata['configuration_label'])
print(f"reference surface temperature: {referencemetadata['surface_temperature_k']:.2f} K")
print(f"reference CAPE: {referencemetadata['cape_jkg']:.0f} J kg-1")
print(f"reference precipitation: {referencemetadata['precipitation_mmday']:.2f} mm day-1")

def makeparams(settings, updates=None):
    params = default_params(device=device)
    params.update(extract_param_overrides(load_run_config()))
    params.update({
        'dt': settings['dt'],
        'ps0': settings['surface_pressure'],
        'ts_init': settings['surface_temperature'],
        'solar_constant': settings['solar_constant'],
        'zenith_factor': settings['zenith_factor'],
        'ocean_depth': settings['ocean_depth'],
        'albedo': settings['surface_albedo'],
        'wind_speed': settings['wind_speed'],
        'convection_scheme': 'mass_flux',
        'radiation_scheme': 'multiband',
        'use_slab_ocean': True,
    })
    if updates is not None:
        params.update(updates)
    return params

def loadreference(nlevels=20, batch=1):
    reference = np.load(root / 'notebooks/data/atm407_equilibrium_20level.npz')
    settings = dict(experiment)
    settings['nlevels'] = nlevels
    grid = make_grid(nlevels, device=device)
    params = makeparams(settings)
    state = initial_state(batch, grid, params, device=device)
    sourcesigma = reference['sigma_full']
    targetsigma = grid['sigma_full'].cpu().numpy()

    for name in ['t', 'q', 'qc', 'cloud_fraction']:
        profile = np.interp(targetsigma, sourcesigma, reference[name])
        values = torch.as_tensor(profile, dtype=state[name].dtype, device=device)
        state[name] = values.unsqueeze(0).repeat(batch, 1)

    referencegrid = make_grid(len(sourcesigma), device=device)
    sourcedsigma = referencegrid['dsigma'].cpu().numpy()
    sourcewater = np.sum(reference['q'] * sourcedsigma)
    targetwater = torch.sum(state['q'][0] * grid['dsigma']).item()
    state['q'] = state['q'] * (sourcewater / targetwater)
    state['ts'].fill_(float(reference['ts']))
    state['ps'].fill_(float(reference['ps']))
    state['slab_ts_ref'] = state['ts'].clone()
    state['slab_energy'].zero_()
    return update_derived(state, grid)

def perturbstate(state, grid, cooling=2.0):
    perturbed = deepcopy(state)
    sigma = grid['sigma_full'].to(perturbed['t'].dtype)
    shape = torch.exp(-0.5 * ((sigma - 0.65) / 0.12) ** 2)
    perturbed['t'] = perturbed['t'] - cooling * shape.unsqueeze(0)
    return update_derived(perturbed, grid)

def integrate(settings, updates=None, batch=1, state=None):
    grid = make_grid(settings['nlevels'], device=device)
    params = makeparams(settings, updates)
    if state is None:
        state = initial_state(batch, grid, params, device=device)
    stepsperday = round(86400 / settings['dt'])
    nsteps = round(settings['days'] * stepsperday)
    diagnosticsteps = max(1, round(settings['diagnostic_hours'] * 3600 / settings['dt']))
    start = time.perf_counter()
    state, history = run(
        state, grid, params, nsteps,
        rad_interval=settings['radiation_steps'],
        diag_interval=diagnosticsteps,
    )
    elapsed = time.perf_counter() - start
    return grid, params, state, history, elapsed

def series(history, name, member=0, scale=1.0):
    values = [entry[name][member].detach().cpu().item() for entry in history]
    return np.array(values) * scale

def days(history, dt):
    return np.array([entry['step'] for entry in history]) * dt / 86400

## Exercise 1: vertical coordinate and column mass

The model uses the terrain-following coordinate $\sigma=p/p_s$. The cell loads the near-equilibrium reference state. Plot full-level pressure and layer pressure thickness. Explain why $\Delta p/g$ is the mass per unit area of a hydrostatic layer. Then use the printed sum to verify that the discrete atmospheric mass is consistent with $p_s/g$.

In [ ]:
grid = make_grid(experiment['nlevels'], device=device)
params = makeparams(experiment)
state = loadreference(experiment['nlevels'])
pressure = state['p'][0].cpu().numpy() / 100
deltap = state['dp'][0].cpu().numpy() / 100
levels = np.arange(experiment['nlevels'])

fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
axes[0].plot(pressure, levels, marker='o')
axes[0].set_xlabel('full-level pressure (hPa)')
axes[1].barh(levels, deltap)
axes[1].set_xlabel('layer pressure thickness (hPa)')
axes[0].set_ylabel('model level')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

massfromlayers = state['dp'].sum().item() / g
massfromsurface = state['ps'].item() / g
print(f'layer sum: {massfromlayers:.2f} kg m-2')
print(f'ps / g:    {massfromsurface:.2f} kg m-2')

## Exercise 2: diagnose the initial sounding

Compute potential temperature and relative humidity. Identify statically stable and weakly stable portions of the column from $\partial\theta/\partial z$. Why does potential temperature, rather than temperature, diagnose dry static stability?

In [ ]:
temperature = state['t'][0]
pressurepa = state['p'][0]
theta = temperature * (100000.0 / pressurepa) ** (Rd / cp)
rh = relative_humidity(state['q'], state['t'], state['p'])[0] * 100

fig, axes = plt.subplots(1, 3, figsize=(11, 5), sharey=True)
axes[0].plot(temperature.cpu(), pressure)
axes[0].set_xlabel('temperature (K)')
axes[1].plot(theta.cpu(), pressure)
axes[1].set_xlabel('potential temperature (K)')
axes[2].plot(rh.cpu(), pressure)
axes[2].set_xlabel('relative humidity (%)')
axes[0].set_ylabel('pressure (hPa)')
axes[0].invert_yaxis()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Exercise 3: follow one physics timestep

A host dynamical core would call the column physics once per timestep. The SCM applies radiation, surface exchange, boundary-layer mixing, shallow convection, deep convection, and condensation sequentially. Run one step and compare each component's contribution to column moist-enthalpy tendency. Large opposing radiation and surface terms can be physically consistent when their sum is small. Which processes redistribute energy internally, and which exchange energy across the top or bottom boundary?

In [ ]:
stepstate = deepcopy(state)
stepstate, diagnostics, radiationcache = physics_step(stepstate, grid, params)
components = [
    'rad_energy_tendency',
    'surface_energy_tendency',
    'bl_energy_tendency',
    'shallow_energy_tendency',
    'conv_energy_tendency',
    'condensation_energy_tendency',
    'cloud_energy_tendency',
]
labels = ['radiation', 'surface', 'boundary layer', 'shallow', 'deep convection', 'condensation', 'clouds']
values = [diagnostics[name][0].item() for name in components]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['tab:red' if value > 0 else 'tab:blue' for value in values]
ax.bar(labels, values, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('column energy tendency (W m-2)')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()
print(f"TOA net flux: {diagnostics['toa_net'][0].item():+.2f} W m-2")
print(f"surface total flux: {diagnostics['surface_total_flux'][0].item():+.2f} W m-2")
print(f"sum of component tendencies: {sum(values):+.2f} W m-2")
print(f"column residual: {diagnostics['column_energy_residual'][0].item():+.2f} W m-2")

## Exercise 4: adjustment after free-tropospheric cooling

Begin from the reference state, impose a 2 K cooling centered near $\sigma=0.65$, and integrate for three days. This represents a temperature tendency that a dynamical core might create through ascent or advection. Predict the CAPE response before running the cell. Does convection remove the resulting instability instantaneously? Separate deep-convective rain from large-scale condensation.

In [ ]:
grid = make_grid(experiment['nlevels'], device=device)
perturbedstate = perturbstate(loadreference(experiment['nlevels']), grid)
grid, params, controlstate, controlhistory, elapsed = integrate(
    experiment, state=perturbedstate
)
timeaxis = days(controlhistory, experiment['dt'])

fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
axes[0].plot(timeaxis, series(controlhistory, 'cape'))
axes[0].set_ylabel('CAPE (J kg-1)')
axes[1].plot(timeaxis, series(controlhistory, 'precip_conv', scale=86400), label='deep convection')
axes[1].plot(timeaxis, series(controlhistory, 'precip_ls', scale=86400), label='large scale')
axes[1].set_ylabel('rain (mm day-1)')
axes[1].legend()
axes[2].plot(timeaxis, series(controlhistory, 'toa_net'))
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set(xlabel='model day', ylabel='TOA net flux (W m-2)')
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
print(f'control runtime: {elapsed:.1f} s')

## Exercise 5: diagnose the mass-flux closure

Starting from the same cooled sounding as Exercise 4, vary entrainment and the requested CAPE-removal timescale for two days. Before running the cases, predict which parameter will matter more. The scheme diagnoses a cloud-base mass flux from CAPE and then limits it with `mf_mb_max`. Compare the unlimited and applied mass fluxes and determine whether the requested `tau_cape` can influence the tendencies in these runs. A parameter appearing in a closure does not guarantee that it controls the realized response when another limiter is active.

All nine columns run simultaneously as a batch to keep the wait short.

In [ ]:
entrainmentvalues = [2.0e-6, 5.0e-6, 1.5e-5]
timescalevalues = [1800.0, 3600.0, 7200.0]
cases = [(entrainment, timescale) for entrainment in entrainmentvalues for timescale in timescalevalues]
updates = {
    'entrainment_rate': torch.tensor([case[0] for case in cases], device=device),
    'tau_cape': torch.tensor([case[1] for case in cases], device=device),
}
challengesettings = dict(experiment)
challengesettings['days'] = 2
challengegrid = make_grid(experiment['nlevels'], device=device)
challengestart = perturbstate(
    loadreference(experiment['nlevels'], batch=len(cases)), challengegrid
)
grid, params, challengestate, challengehistory, elapsed = integrate(
    challengesettings, updates=updates, batch=len(cases), state=challengestart,
)
reductions = []
meanrain = []
capfractions = []
appliedmassflux = []
unlimitedmassflux = []
for member, case in enumerate(cases):
    cape = series(challengehistory, 'cape', member=member)
    reductions.append(cape[0] - cape[-1])
    rain = series(challengehistory, 'precip_total', member=member, scale=86400)
    meanrain.append(rain.mean())
    capfractions.append(series(challengehistory, 'mass_flux_cap_active', member=member).mean())
    appliedmassflux.append(series(challengehistory, 'cloud_base_mass_flux', member=member).mean())
    unlimitedmassflux.append(series(challengehistory, 'cloud_base_mass_flux_unlimited', member=member).mean())

resultgrid = np.array(reductions).reshape(len(entrainmentvalues), len(timescalevalues))
capgrid = np.array(capfractions).reshape(len(entrainmentvalues), len(timescalevalues))
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
images = [
    axes[0].imshow(resultgrid, origin='lower', cmap='Blues'),
    axes[1].imshow(capgrid, origin='lower', vmin=0, vmax=1, cmap='Oranges'),
]
for ax in axes:
    ax.set_xticks(range(len(timescalevalues)), [f'{value / 3600:.1f}' for value in timescalevalues])
    ax.set_yticks(range(len(entrainmentvalues)), [f'{value:.1e}' for value in entrainmentvalues])
    ax.set(xlabel='requested CAPE timescale (hours)', ylabel='entrainment rate (Pa-1)')
axes[0].set_title('two-day CAPE reduction')
axes[1].set_title('fraction of diagnostics at mass-flux cap')
fig.colorbar(images[0], ax=axes[0], label='CAPE reduction (J kg-1)')
fig.colorbar(images[1], ax=axes[1], label='cap-active fraction')
plt.show()

winner = int(np.argmax(reductions))
print('largest CAPE reduction (entrainment, timescale):', cases[winner])
print(f'CAPE reduction: {reductions[winner]:.1f} J kg-1')
print(f'mean total rain: {meanrain[winner]:.2f} mm day-1')
print(f'mean applied mass flux: {appliedmassflux[winner]:.3f} kg m-2 s-1')
print(f'mean unlimited mass flux: {unlimitedmassflux[winner]:.1f} kg m-2 s-1')
print(f'cap-active fraction: {capfractions[winner]:.2f}')
print(f'batched challenge runtime: {elapsed:.1f} s')

## Exercise 6: numerical sensitivity

Interpolate the 20-level reference state to 10, 20, and 40 levels, then give each grid one day to adjust. Compare CAPE before and after adjustment, precipitation, and runtime. Surface coupling, boundary-layer depth, subcloud export, plume entrainment, and plume detrainment are expressed in resolution-aware pressure or sigma coordinates, but an interpolated state is still not a native-grid equilibrium. This is therefore a remapping stress test rather than a formal convergence test. Explain why CAPE can still change when the same sounding is sampled on another grid, and describe the separate native-grid integrations that would be required for a convergence claim.

In [ ]:
resolutionresults = []
for nlevels in [10, 20, 40]:
    settings = dict(experiment)
    settings['nlevels'] = nlevels
    settings['days'] = 1
    initialstate = loadreference(nlevels)
    diagnosticstate = deepcopy(initialstate)
    diagnosticgrid = make_grid(nlevels, device=device)
    diagnosticparams = makeparams(settings)
    diagnosticstate, initialdiagnostics, cache = physics_step(
        diagnosticstate, diagnosticgrid, diagnosticparams
    )
    grid, params, finalstate, history, elapsed = integrate(
        settings, state=initialstate
    )
    resolutionresults.append({
        'levels': nlevels,
        'initialcape': initialdiagnostics['cape'][0].item(),
        'cape': series(history, 'cape')[-1],
        'rain': series(history, 'precip_total', scale=86400).mean(),
        'runtime': elapsed,
    })

for result in resolutionresults:
    print(
        f"{result['levels']:2d} levels | "
        f"CAPE {result['initialcape']:7.1f} -> {result['cape']:7.1f} J kg-1 | "
        f"mean rain {result['rain']:5.2f} mm day-1 | "
        f"runtime {result['runtime']:4.1f} s"
    )

## Submission

Submit your completed notebook with: (1) a physical explanation for each result, (2) your prediction and interpretation for the challenge, (3) your numerical-convergence criterion, and (4) one paragraph explaining what a single-column model cannot represent without a dynamical core.